# Bank Management System Database Setup & Analysis (1,500 Accounts)

This Jupyter Notebook creates and populates a relational **Bank Management System Database** using SQLite, updated to generate **1,500 rows in the `Accounts` table**.

### Database Architecture & Key Criteria
- **Tables**:
  1. `Branches`: Core branch directory across regions.
  2. `Customers`: Customer demographic and account owner details (500 customers).
  3. `Accounts`: **1,500 Bank Accounts** linked to customers and branches via foreign keys.
  4. `Transactions`: Financial records connected to accounts.
- **Key Features**: Primary Keys, Foreign Key relationships, `CHECK` domain constraints.
- **Data Types & Scales**:
  - **Nominal**: Customer names, account types (`Checking`, `Savings`, `Money Market`), transaction types.
  - **Ordinal**: Customer risk ratings (`Low`, `Medium`, `High`), Branch tier levels.
  - **Interval**: Transaction timestamps, Account opening dates.
  - **Ratio**: Account balances, Transaction amounts, Interest rates (%).

In [ ]:
# Step 1: Database Initialization and Schema Creation
import sqlite3
import os

DB_NAME = 'bank_management.db'

if os.path.exists(DB_NAME):
    os.remove(DB_NAME)

conn = sqlite3.connect(DB_NAME)
cur = conn.cursor()
cur.execute("PRAGMA foreign_keys = ON;")

cur.executescript("""
-- TABLE 1: BRANCHES
CREATE TABLE Branches (
    branch_id INTEGER PRIMARY KEY AUTOINCREMENT,
    branch_name TEXT NOT NULL,
    city TEXT NOT NULL,
    state TEXT NOT NULL,
    branch_tier TEXT NOT NULL CHECK (branch_tier IN ('Main', 'Regional', 'Sub-Branch'))
);

-- TABLE 2: CUSTOMERS
CREATE TABLE Customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    phone TEXT,
    risk_rating TEXT NOT NULL CHECK (risk_rating IN ('Low', 'Medium', 'High')),
    created_at TEXT NOT NULL
);

-- TABLE 3: ACCOUNTS (1,500 rows target)
CREATE TABLE Accounts (
    account_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    branch_id INTEGER NOT NULL,
    account_type TEXT NOT NULL CHECK (account_type IN ('Savings', 'Checking', 'Money Market')),
    balance REAL NOT NULL CHECK (balance >= 0),
    interest_rate REAL NOT NULL CHECK (interest_rate >= 0),
    opened_date TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id),
    FOREIGN KEY (branch_id) REFERENCES Branches(branch_id)
);

-- TABLE 4: TRANSACTIONS
CREATE TABLE Transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    account_id INTEGER NOT NULL,
    transaction_type TEXT NOT NULL CHECK (transaction_type IN ('Deposit', 'Withdrawal', 'Transfer')),
    amount REAL NOT NULL CHECK (amount > 0),
    transaction_date TEXT NOT NULL,
    FOREIGN KEY (account_id) REFERENCES Accounts(account_id)
);
""")

conn.commit()
print("Database tables created successfully with foreign key and domain constraints.")

In [ ]:
# Step 2: Data Population (Populating Branches, Customers, 1,500 Accounts, and Transactions)
import random
from datetime import datetime, timedelta

random.seed(42)

# 1. Insert Branches
branches = [
    ('Central Hub', 'London', 'Greater London', 'Main'),
    ('Financial District Branch', 'London', 'Greater London', 'Main'),
    ('North Regional', 'Manchester', 'Greater Manchester', 'Regional'),
    ('Midlands Express', 'Birmingham', 'West Midlands', 'Regional'),
    ('Scotland Central', 'Edinburgh', 'Midlothian', 'Regional'),
    ('Quayside Branch', 'Newcastle', 'Tyne and Wear', 'Sub-Branch')
]
cur.executemany("INSERT INTO Branches (branch_name, city, state, branch_tier) VALUES (?, ?, ?, ?);", branches)

# 2. Insert Customers (500 records)
first_names = ['James', 'Emma', 'Liam', 'Olivia', 'Noah', 'Ava', 'Ethan', 'Sophia', 'Mason', 'Isabella']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Rodriguez', 'Martinez']
risk_levels = ['Low', 'Medium', 'High']
risk_weights = [0.7, 0.25, 0.05]

customers = []
for i in range(1, 501):
    fn = random.choice(first_names)
    ln = random.choice(last_names)
    email = f"{fn.lower()}.{ln.lower()}{i}@examplebank.com"
    phone = f"+44 7700 900{i:03d}"
    risk = random.choices(risk_levels, weights=risk_weights)[0]
    created = (datetime(2023, 1, 1) + timedelta(days=random.randint(0, 365))).strftime('%Y-%m-%d')
    customers.append((fn, ln, email, phone, risk, created))

cur.executemany("INSERT INTO Customers (first_name, last_name, email, phone, risk_rating, created_at) VALUES (?, ?, ?, ?, ?, ?);", customers)

# 3. Insert Accounts (EXACTLY 1,500 ROWS across 500 customers)
account_types = ['Savings', 'Checking', 'Money Market']
accounts = []
for i in range(1, 1501):
    cust_id = random.randint(1, 500)
    branch_id = random.randint(1, len(branches))
    acct_type = random.choice(account_types)
    balance = round(random.uniform(500, 75000), 2)
    interest = 0.05 if acct_type == 'Savings' else (0.02 if acct_type == 'Money Market' else 0.00)
    opened = (datetime(2024, 1, 1) + timedelta(days=random.randint(0, 300))).strftime('%Y-%m-%d')
    accounts.append((cust_id, branch_id, acct_type, balance, interest, opened))

cur.executemany("INSERT INTO Accounts (customer_id, branch_id, account_type, balance, interest_rate, opened_date) VALUES (?, ?, ?, ?, ?, ?);", accounts)

# 4. Insert Transactions (2,000 records across 1,500 accounts)
tx_types = ['Deposit', 'Withdrawal', 'Transfer']
tx_weights = [0.5, 0.35, 0.15]
transactions = []

base_date = datetime(2025, 1, 1)
for i in range(2000):
    acct_id = random.randint(1, 1500)
    tx_type = random.choices(tx_types, weights=tx_weights)[0]
    amount = round(random.uniform(10, 2500), 2)
    tx_date = (base_date + timedelta(days=random.randint(0, 300), hours=random.randint(0, 23))).strftime('%Y-%m-%d %H:%M:%S')
    transactions.append((acct_id, tx_type, amount, tx_date))

cur.executemany("INSERT INTO Transactions (account_id, transaction_type, amount, transaction_date) VALUES (?, ?, ?, ?);", transactions)

conn.commit()
print(f"Database populated with {len(branches)} branches, {len(customers)} customers, {len(accounts)} accounts, and {len(transactions)} transactions.")

## Analytical Queries & Verification

Executing SQL queries to verify row counts and analyze the 1,500 accounts.

In [ ]:
import pandas as pd

# 1. Verify Accounts Row Count
q_count = pd.read_sql_query("SELECT COUNT(*) AS total_accounts FROM Accounts;", conn)
print("=== Query 1: Total Accounts Count ===")
display(q_count)

# 2. Summary by Account Type
q_summary = pd.read_sql_query("""
SELECT 
    account_type,
    COUNT(*) AS account_count,
    ROUND(SUM(balance), 2) AS total_balance,
    ROUND(AVG(balance), 2) AS avg_balance
FROM Accounts
GROUP BY account_type
ORDER BY total_balance DESC;
""", conn)
print("\n=== Query 2: Summary by Account Type ===")
display(q_summary)

# 3. Accounts Distribution by Branch
q_branch = pd.read_sql_query("""
SELECT 
    b.branch_name,
    b.city,
    COUNT(a.account_id) AS accounts_count,
    ROUND(SUM(a.balance), 2) AS total_deposited_balance
FROM Branches b
JOIN Accounts a ON b.branch_id = a.branch_id
GROUP BY b.branch_id
ORDER BY accounts_count DESC;
""", conn)
print("\n=== Query 3: Branch Account Metrics ===")
display(q_branch)

# 4. Sample Accounts Rows
q_sample = pd.read_sql_query("SELECT * FROM Accounts LIMIT 10;", conn)
print("\n=== Query 4: Sample 10 Rows from Accounts ===")
display(q_sample)

conn.close()